# Retrieval Runner

Runs one config against one query set, using the index built by `index_builder.ipynb`.

**What changed in S1.** The run is identified by `(config, query set)`, not by a method name
that had to match a folder. The retriever module is read from the config's `retriever.module`
— the in-notebook alias table is gone (D-016), so renaming a config no longer changes which
code runs. And the gold is read from the query record rather than inferred from the query's
own key: a BC3CAT-Syn query is keyed `<leaf>_syn_<hash>` and its gold is `gold_item_key`, so
the old assumption would have scored every synthetic query against a document that does not
exist, and dropped it without a word.

Results are written to `runs/{collection}/{queryset}/{method}` (D-008).


In [ ]:
# papermill parameters
CFG_PATH      = "/work/configs/bm25_unigram_params__k1-0.60__b-0.35__OE.yaml"
QUERYSET      = "single_texto"   # texto | resumen | single_texto | stacked_texto
SPLIT         = "dev"            # dev | test | all — test is one-shot, in S12 only (D-007)
WORK_ROOT     = "/work"
RANDOM_SAMPLE = None             # None = every query; an int samples with random_state=42
MAX_QUERIES   = None
SAVE          = True
K             = 100
BATCH_SIZE    = None             # None = the retriever's own default
RUN_CONTEXT   = "manual"


In [ ]:
# === retrieve.ipynb — one config, one query set, gold from the record ===

import gzip
import json
import sys
import time
from importlib import import_module
from pathlib import Path

for _candidate in (Path.cwd(), Path(WORK_ROOT) / "src"):
    if (_candidate / "utils").is_dir() and str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))

import numpy as np
import pandas as pd
from IPython.display import display

from utils.corpus_prep import assert_gold_present
from utils.provenance import provenance_stamp
from utils.run_context import load_run_context
from utils.splits import select_split

ctx = load_run_context(CFG_PATH, queryset=QUERYSET, work_root=WORK_ROOT)

print(f"config    : {ctx.config_path.name}")
print(f"collection: {ctx.collection} | query set: {ctx.queryset} | split: {SPLIT}")
print(f"index     : {ctx.index_dir}")
print(f"queries   : {ctx.query_path}")
print(f"run dir   : {ctx.run_dir}")


def read_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_frame(path: Path) -> pd.DataFrame:
    if path.suffix.lower() in (".json", ".jsonl"):
        try:
            return pd.read_json(path, lines=True)
        except ValueError:
            return pd.read_json(path)
    return pd.read_parquet(path)


# -------------- the index --------------
for required in ("meta.json", "fields.json", "mapping.jsonl", "data"):
    if not (ctx.index_dir / required).exists():
        raise FileNotFoundError(
            f"missing {required} in {ctx.index_dir}. Build the index first: index_builder.ipynb "
            f"with CFG_PATH={ctx.config_path}"
        )

meta = read_json(ctx.index_dir / "meta.json")
fields = read_json(ctx.index_dir / "fields.json")
print(f"\nindex     : {meta.get('method')} / {meta.get('impl')} | docs={meta.get('num_docs')} "
      f"| text_field={meta.get('text_field')}")

if meta.get("collection") != ctx.collection:
    raise ValueError(
        f"index was built for collection {meta.get('collection')!r} but this config says "
        f"{ctx.collection!r}. Indexes are per collection (D-008)."
    )

# -------------- the queries --------------
queries_df = read_frame(ctx.query_path)

text_field = meta.get("text_field") or fields.get("text_field")
if text_field not in queries_df.columns:
    raise KeyError(
        f"the index was built on {text_field!r}, which {ctx.query_path.name} does not have. "
        f"Available: {sorted(queries_df.columns)[:12]}"
    )

# Gold is declared by the query, never inferred from its key (INTAKE §4 breakpoint 1).
gold_column = "gold_item_key" if "gold_item_key" in queries_df.columns else "item_key"
print(f"gold      : {gold_column}")

# The index's own external ids are the corpus as the run sees it.
mapping_keys = set()
with open(ctx.index_dir / "mapping.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            mapping_keys.add(str(json.loads(line)["external_id"]))

# Fail loud, before anything is scored: a query whose gold is not in the index is never
# silently dropped, because a dropped query shrinks the denominator instead of scoring zero.
assert_gold_present(queries_df.assign(**{gold_column: queries_df[gold_column].astype(str)}), mapping_keys)

n_before = len(queries_df)
queries_df = select_split(queries_df, SPLIT)
n_after_split = len(queries_df)
print(f"queries   : {len(queries_df)} of {n_before} after split={SPLIT}")

if RANDOM_SAMPLE is not None:
    queries_df = queries_df.sample(n=int(RANDOM_SAMPLE), random_state=42).copy()
    print(f"           sampled {len(queries_df)} with random_state=42")
if MAX_QUERIES is not None:
    queries_df = queries_df.head(int(MAX_QUERIES)).copy()
if len(queries_df) == 0:
    raise ValueError(f"no queries left after split={SPLIT}; nothing to run")

# What entered scoring, after every filter. Anything less coming out is a dropped query.
n_selected = len(queries_df)

# -------------- the retriever --------------
# From the config, not from the filename (D-016).
retr_mod = import_module(ctx.retriever_module)
entrypoint = getattr(retr_mod, ctx.retriever_entrypoint, None)
if entrypoint is None:
    raise AttributeError(
        f"{ctx.retriever_module} exposes no {ctx.retriever_entrypoint}(index_dir)"
    )
searcher = entrypoint(ctx.index_dir)
print(f"retriever : {ctx.retriever_module}.{ctx.retriever_entrypoint}")
# A retriever that reads the query record, not only its text, declares `bind_queries` and gets
# the selected rows in scoring order (S5: the structured oracle-extraction bound). No other
# retriever has it, so for them this line does nothing.
if hasattr(searcher, "bind_queries"):
    searcher.bind_queries(queries_df)

# -------------- run --------------
texts = queries_df[text_field].astype(str).tolist()
qkeys = queries_df["item_key"].astype(str).tolist()
gold_keys = queries_df[gold_column].astype(str).tolist()
gold_parents = (
    queries_df["parent_key"].astype(str).tolist()
    if "parent_key" in queries_df.columns
    else [""] * len(qkeys)
)

t0 = time.time()
search_kwargs = {} if BATCH_SIZE is None else {"batch_size": BATCH_SIZE}
top_idx, top_scores = searcher.search_batch(texts, k=K, **search_kwargs)
elapsed = time.time() - t0

if len(top_idx) != len(qkeys):
    raise RuntimeError(
        f"the retriever returned {len(top_idx)} result rows for {len(qkeys)} queries; "
        "no query is dropped between input and output"
    )

ext_ids = np.asarray(searcher.external_ids, dtype=object)
top_ext = ext_ids[top_idx]
gold_arr = np.asarray(gold_keys, dtype=object)[:, None]

hits_at1 = (top_ext[:, :1] == gold_arr).any(axis=1)
hits_at5 = (top_ext[:, :5] == gold_arr).any(axis=1) if K >= 5 else hits_at1
hits_at10 = (top_ext[:, :10] == gold_arr).any(axis=1) if K >= 10 else hits_at5

print(
    f"\nitem-level on {len(qkeys)} queries — Acc@1={hits_at1.mean():.4f} | "
    f"Recall@5={hits_at5.mean():.4f} | Recall@10={hits_at10.mean():.4f}"
)
print(f"elapsed   : {elapsed:.2f}s (B={len(qkeys)}, K={K})")
print("These are the runner's own quick figures. The reported metrics come from metrics.ipynb.")

# -------------- preview --------------
preview_rows = []
for i in range(min(5, len(qkeys))):
    for r in range(min(5, top_idx.shape[1])):
        doc_i = int(top_idx[i, r])
        preview_rows.append({
            "rank": r + 1,
            "query_item_key": qkeys[i],
            "gold_item_key": gold_keys[i],
            "index_item_key": str(ext_ids[doc_i]),
            "score": float(top_scores[i, r]),
            "hit": gold_keys[i] == str(ext_ids[doc_i]),
        })
display(pd.DataFrame(preview_rows))

# -------------- save --------------
if SAVE:
    ctx.run_dir.mkdir(parents=True, exist_ok=True)
    out_path = ctx.run_dir / f"results_top{K}.jsonl.gz"
    with gzip.open(out_path, "wt", encoding="utf-8") as gz:
        for i, (qkey, qtext) in enumerate(zip(qkeys, texts)):
            candidates = [
                {
                    "rank": r + 1,
                    "doc_id": int(top_idx[i, r]),
                    "index_item_key": str(ext_ids[int(top_idx[i, r])]),
                    "score": float(top_scores[i, r]),
                }
                for r in range(top_idx.shape[1])
            ]
            gz.write(json.dumps({
                "query_id": f"q_{i:06d}",
                "query_item_key": qkey,
                # Carried explicitly: metrics.ipynb must never re-infer the gold.
                "gold_item_key": gold_keys[i],
                "gold_parent_key": gold_parents[i],
                "query_text": qtext,
                "candidates": candidates,
                "score_info": {"family": "cosine", "higher_is_better": True},
            }, ensure_ascii=False) + "\n")

    # The four-part stamp, computed from the files actually read (operating rule 1).
    # A report quotes this; it never assembles it.
    run_meta = {
        **provenance_stamp(
            run_id=f"{ctx.collection}/{ctx.queryset}/{ctx.method}",
            config_path=ctx.config_path,
            query_path=ctx.query_path,
            repo=Path(WORK_ROOT),
        ),
        "corpus_sha256": None,  # filled below, once, from the corpus the index was built on
        "collection": ctx.collection,
        "queryset": ctx.queryset,
        "split": SPLIT,
        "method": ctx.method,
        "config": str(ctx.config_path),
        "index_dir": str(ctx.index_dir),
        "query_path": str(ctx.query_path),
        "gold_column": gold_column,
        "queries": len(qkeys),
        "queries_before_split": int(n_before),
        "queries_after_split": int(n_after_split),
        # After every filter, including the sample: the population that entered scoring.
        "queries_selected": int(n_selected),
        # Measured, not asserted: the count that entered scoring against the count that came
        # out. It is necessarily 0 given the raise above, but a literal 0 is not evidence and
        # the test that checked it could not fail (audit F4).
        "dropped": int(n_selected - len(qkeys)),
        "K": K,
        "batch_size": BATCH_SIZE,
        "random_sample": RANDOM_SAMPLE,
        "random_state": 42 if RANDOM_SAMPLE is not None else None,
        "retriever_module": ctx.retriever_module,
        "elapsed_s": round(elapsed, 3),
        "created_utc": pd.Timestamp.utcnow().isoformat(),
    }
    from utils.provenance import sha256_file
    run_meta["corpus_sha256"] = sha256_file(ctx.corpus_path)
    with open(ctx.run_dir / "run_meta.json", "w", encoding="utf-8") as f:
        json.dump(run_meta, f, ensure_ascii=False, indent=2)

    print(f"\nsaved {out_path}")
    print(f"saved {ctx.run_dir / 'run_meta.json'}")
else:
    print("\nNot saving (SAVE is False).")
